In [70]:
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)
import tensorflow as tf
from sklearn import preprocessing
import numpy as np
import random
import matplotlib.pyplot as plt
import sys
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from matplotlib import cm  
from matplotlib.patches import ConnectionPatch
from AES import*
import pandas as pd
from Model.CNN import cnn_classifier
from utils.LoadData import load_CW_Source,load_CW_Target
from  numba  import njit, prange
from sklearn.utils import shuffle

In [ ]:
@njit
def plot_guessing_entropy_Key(P_VBox,preds, real_diff, plaintext,trace_num_max,type):

    """
    - preds : the probability for each class (n*256 for a byte, n*9 for Hamming weight)
    - real_key : the key of the target device
    - device_id : id of the target device
    - model_flag : a string for naming GE result
    """
    # GE/SR is averaged over 20 attacks 
    num_averaged = 20
    # max trace num for attack
    guessing_entropy = np.zeros((num_averaged, trace_num_max))
    success_flag = np.zeros((num_averaged, trace_num_max))
    
    # attack multiples times for average
    for time in range(num_averaged):
        # select the attack traces randomly
        random_index_total = list(range(plaintext.shape[0]))
        random_index_total = np.random.permutation(np.arange(plaintext.shape[0]))
        random_index = random_index_total[0:trace_num_max]
        # initialize score matrix
        score_mat = np.zeros((trace_num_max, 256))
        for ps in range(0, 256):
            vs=P_VBox[ps]
            for i in range(trace_num_max):
                diff = plaintext[random_index[i]] ^ ps
                diff = np.int64(plaintext[random_index[i]] ^ ps)
                vs = np.int64(P_VBox[ps])
                score_mat[i, diff] = preds[random_index[i], vs]

        for i in range(0, trace_num_max):
            log_likelihood = np.sum(score_mat[0:i+1,:], axis=0)
            ranked = np.argsort(log_likelihood)[::-1]
            guessing_entropy[time,i] =  list(ranked).index(real_diff)
            if list(ranked).index(real_diff) == 0:
                    success_flag[time, i] = 1
 
    guessing_entropy_avg = np.zeros(trace_num_max)
    for i in range(trace_num_max):
        guessing_entropy_avg[i] = np.sum(guessing_entropy[:, i]) / num_averaged

    success_flag = np.sum(success_flag, axis=0)
    success_rate = success_flag/num_averaged 
    return  guessing_entropy_avg,success_rate

In [ ]:
# Main parameter initialization
profiling_Data_path='../Dataset/mask_AES/AES_Sbox/'
Target_Data_path='../Dataset/mask_AES/SM4_Sbox/'
# Target_Data_path='./Dataset/mask_AES/Skinny_Sbox/'
model_path = '../Model/mask_AES/'

In [ ]:
prediction_byte=[]
p_total=[]
for byte in range(16):
    # Load profiling traces (template attack dataset)
    profiling_traces, _, _, _, _, _ = load_CW_Source(
        in_file=profiling_Data_path,
        sec=45000,  # Fixed security parameter from original implementation
        byte=byte
    )

    # Load target device data
    X_attack, label_V, p_attack = load_CW_Target(
        in_file=Target_Data_path,
        byte=byte
    )

    # Preprocessing pipeline
    # 1. Standardization (zero-mean, unit-variance)
    scaler = preprocessing.StandardScaler()
    profiling_traces = scaler.fit_transform(profiling_traces)
    X_attack = scaler.transform(X_attack)

    # # 2. Normalization (scale to [0,1] range)
    scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))
    profiling_traces = scaler.fit_transform(profiling_traces)
    X_attack = scaler.transform(X_attack)
    X_attack = X_attack.reshape(X_attack.shape)

    # load model
    model = cnn_classifier(input_size=600)
    model_name = f'Source_Model_byte{byte}_D1.h5'
    model.load_weights(model_path + model_name)
    predictions = model.predict(X_attack)
    prediction_byte.append(predictions)
    p_total.append(p_attack)

In [76]:
def get_pv_box(models, profiling_Data_path, Target_Data_path, p_v_box,t_num):
    # Main experiment loop (single iteration: 10000 traces only)
    # for t_num in range(500, 2001, 2000):  # Range generates [10000] only
        # Byte position iteration (0-15 for 16-byte blocks)
    for byte in range(0, 16):
            # Load profiling traces (template attack dataset)
            profiling_traces, _, _, _, _, _ = load_CW_Source(
                in_file=profiling_Data_path,
                sec=45000,  # Fixed security parameter from original implementation
                byte=byte
            )
            # Load attack traces and associated data
            X_attack, label_V, p_attack = load_CW_Target(
                in_file=Target_Data_path,
                byte=byte
            )

            # Slice datasets with shuffling 
            X_attack, label_V, p_attack = shuffle(X_attack, label_V, p_attack)

            
            X_attack_shuffle = X_attack[:t_num]  # Use first t_num traces
            label_V_shuffle = label_V[:t_num]
            p_attack_shuffle = p_attack[:t_num]
            

            # Preprocessing pipeline
            # 1. Standardization (zero-mean, unit-variance)
            scaler = preprocessing.StandardScaler()
            profiling_traces = scaler.fit_transform(profiling_traces)
            X_attack_shuffle = scaler.transform(X_attack_shuffle)
            
            # # 2. Normalization (scale to [0,1] range)
            scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))
            profiling_traces = scaler.fit_transform(profiling_traces)
            X_attack_shuffle = scaler.transform(X_attack_shuffle)

            # Model inference using pre-trained byte-specific model
            predictions= models[byte].predict(X_attack_shuffle)  # models loaded externally

            # Process prediction results
            A = np.squeeze(predictions)  # Remove singleton dimensions
            B = np.squeeze(p_attack_shuffle)
            df = pd.DataFrame({'traces': list(A), 'plaintext': list(B)})  # Maintain column names
            
            # Aggregate predictions by plaintext value
            sum_by_plaintext = df.groupby('plaintext')['traces'].sum()
            
            # Populate key guess matrix
            for j in range(256):  # All possible byte values (0-255)
                if j in sum_by_plaintext.index:  # Check if plaintext exists in data
                    # Store index with maximum prediction sum
                    p_v_box[byte, j] = np.argmax(sum_by_plaintext.loc[j])
                else:
                    p_v_box[byte, j] = -1  # Mark missing entries


In [ ]:
GE_inf=[]
k_diff_true = np.zeros([16], dtype=int)
key=[0x3F,0x1C,0x77,0xC5,0xA8,0x6E,0x5A,0xF1,0x19,0xA4,0x07,0x3F,0x51,0xFD,0xAE,0xA7]
for i in range(16):
        k_diff_true[i] = key[4] ^ key[i]  # XOR with first key byte
p_v_box = np.zeros((16, 256)) 
models = []
for byte in range(0,16):
    model = cnn_classifier(input_size=600)
    model_name = f'Source_Model_byte{byte}_D1.h5'
    model.load_weights(model_path + model_name)
    models.append(model)  

get_pv_box(models, profiling_Data_path, Target_Data_path, p_v_box,t_num=1000)
P_VBox=np.zeros(256)
# mapping = p_attack_to_label_total[0]
for i in range(256):
    P_VBox[i] = p_v_box[4][i]


for byte in range(0,16):
    trace_num_max_t=1000

    predictions_t=prediction_byte[byte]
    p_attack=p_total[byte]
    guessing_entropy,success_rate=plot_guessing_entropy_Key(P_VBox,predictions_t, k_diff_true[byte], p_attack,trace_num_max_t,'SKinny')
    # guessing_entropy,success_rate=plot_guessing_entropy_diff(predictions_s,predictions_t, k_diff_true[byte],p_s, p_attack,trace_num_max_s,trace_num_max_t,'SM4')
    GE_inf.append(guessing_entropy)


In [ ]:
# Visualization parameters setup
sec = 1000  # Number of points to display on X-axis
x = np.arange(0, sec)  # Create array for X-axis values
cmap = cm.get_cmap("tab20", 16)  # Get colormap with 16 distinct colors
fig, ax = plt.subplots()  # Create main figure and axes
markers = ['o', 's', 'D', '^', 'v', 'p', '*', 'X', '+', '>', '<', '8', 'H', '1', '2','3']  # Marker styles for different bytes

# Plot Guessing Entropy curves for all 16 bytes
for byte in range(16):
    # Plot each byte's GE curve with unique color from colormap
    ax.plot(x, GE_inf[byte][:sec], label=f'$\Delta$ Byte{byte}', color=cmap(byte))

# Configure main plot appearance
ax.set_xlabel('Number of traces', fontsize=16)
ax.set_ylabel(r'Guessing Entropy', fontsize=16)
ax.tick_params(labelsize=14)
ax.legend(fontsize=8.2)  # Display legend with byte labels

# Create inset axes for zoomed-in view
axins = inset_axes(ax, width="40%", height="30%", loc="lower left",
                   bbox_to_anchor=(0.3, 0.2, 1, 1), bbox_transform=ax.transAxes)

# Plot zoomed-in view of GE curves
for byte in range(16):
    axins.plot(x, GE_inf[byte][:sec], color=cmap(byte))

# Define zoom region (X-axis range 2-10)
zone_left = 5
zone_right =30

# Calculate dynamic axis limits for inset plot
x_ratio = 0.2  # X-axis expansion ratio
y_ratio = 0.2  # Y-axis expansion ratio

# X-axis limits with expansion buffer
xlim0 = x[zone_left] - (x[zone_right] - x[zone_left]) * x_ratio
xlim1 = x[zone_right] + (x[zone_right] - x[zone_left]) * x_ratio

# Y-axis limits based on data range in zoom region
y = np.hstack([GE_inf[byte][zone_left:zone_right] for byte in range(16)])
ylim0 = np.min(y) - (np.max(y) - np.min(y)) * y_ratio
ylim1 = np.max(y) + (np.max(y) - np.min(y)) * y_ratio

# Apply calculated limits to inset plot
axins.set_xlim(xlim0, xlim1)
axins.set_ylim(ylim0, ylim1)

# Create connection lines between main plot and inset
x_main = 30  # Reference X-position for connection lines
y_main = GE_inf[0][x_main]  # Y-value at reference position

# Get inset plot boundary coordinates
x_inset_left = axins.get_xlim()[0]
y_inset_top = axins.get_ylim()[1]
x_inset_right = axins.get_xlim()[1]
y_inset_bottom = axins.get_ylim()[0]

# Create dashed connection lines
line1 = ConnectionPatch(xyA=(x_main, y_main), xyB=(x_inset_left, y_inset_top),
                        coordsA='data', coordsB='data', axesA=ax, axesB=axins,
                        color='red', linestyle='--', linewidth=2)

line2 = ConnectionPatch(xyA=(x_main, y_main), xyB=(x_inset_right, y_inset_bottom),
                        coordsA='data', coordsB='data', axesA=ax, axesB=axins,
                        color='red', linestyle='--', linewidth=2)

# Add connection lines to main plot
ax.add_artist(line1)
ax.add_artist(line2)

# Final plot formatting and saving
ax.grid(True)
plt.grid(True)
plt.show()  # Display interactive plot